In [ ]:
from dataclasses import dataclass
import numpy as np

# Trajectory Container Tools - Direct Instanciation Usage Example

This notebook demonstrates the essential usage of Trajectory Container Tools (TCT) dataclasses for creating and working with trajectory data.

## Import trajectory-container-tools namespace


In [ ]:
import trajectory_container_tools as tct

## 1. Basic Trajectory Creation

### Simple Trajectory Container Structure

The most straightforward way to create a trajectory is by extending `BaseTrajectoryFeature`:

In [ ]:
# Generate sample trajectory data
timesteps = 50
time_array = np.linspace(0, 5, timesteps)
x_positions = np.sin(time_array)
y_positions = np.cos(time_array)
timestamps = np.arange(timesteps) * 0.1  # 10 Hz sampling

@dataclass()
class Simple2DCoordinateTrajectory(tct.BaseTrajectoryFeature):
    x: np.ndarray
    y: np.ndarray
    timestamps: np.ndarray

# Create trajectory container
trajectory = Simple2DCoordinateTrajectory(
    feature_name="2D coordinate",
    x=x_positions,
    y=y_positions,
    timestamps=timestamps
)

print((
        f"Created trajectory with {trajectory.trajectory_len} timesteps\n"
        f"Available dimensions: {trajectory.get_public_attribute_names()}\n"
), trajectory)


### Nested Trajectory Container Structure

For more complex data organization, you can nest trajectory dataclasses. Check dataclasses from `tct.dataclasses` module for inspiration:

In [ ]:
from trajectory_container_tools.dataclasses import Vector2D

@dataclass()
class CustomPoseContainer(tct.BaseTrajectoryFeature):
    x: np.ndarray
    y: np.ndarray


@dataclass()
class ComplexTrajectory(tct.BaseTrajectoryFeature):
    timestamps: np.ndarray
    position: CustomPoseContainer
    velocity: Vector2D


sin_cos_trajectory_object = ComplexTrajectory(feature_name="mock",
                                              timestamps=timestamps,
                                              position=CustomPoseContainer(x_positions, y_positions),
                                              velocity=Vector2D(np.ones_like(x_positions), np.ones_like(y_positions))
                                              )

print(sin_cos_trajectory_object)

## 2. Factory-Based Creation

TCT provides factory functions for dynamic trajectory dataclass creation:

In [ ]:
mock_data = np.random.randn(100, 4)  # 100 timesteps, 4 dimensions

# Define the specification
spec = tct.factory.TrjDataClassFeatureSpecification(
        new_feature_dataclass_type='DynamicTrajectory',
        dimension_names=('x', 'y', 'velocity', 'acceleration')
        )

# Create the dataclass type
DynamicTrajectory = tct.factory.create_dataclass(specification=spec)

# Use the dynamically created class
factory_generated_trajectory = DynamicTrajectory(
        feature_name="Factory-made-mock-trajectory",
        x=mock_data[:, 0],
        y=mock_data[:, 1],
        velocity=mock_data[:, 2],
        acceleration=mock_data[:, 3]
        )

print(factory_generated_trajectory)

## 3. Accessing Trajectory Data

Demonstrate basic data access and manipulation.


In [ ]:
# Access individual dimensions
print(f"X position range: [{ trajectory.x.min():.2f}, { trajectory.x.max():.2f}]")
print(f"Y position range: [{ trajectory.y.min():.2f}, { trajectory.y.max():.2f}]")
print(f"Time range: [{ trajectory.timestamps.min():.2f}, { trajectory.timestamps.max():.2f}] seconds")

## 4. Trajectory Slicing

Extract portions of the trajectory.


In [ ]:
# Slice trajectory (get timesteps 10-30)
partial_trajectory = trajectory[10:30]
print(f"Original trajectory length: {trajectory.trajectory_len}")
print(f"Partial trajectory length: {partial_trajectory.trajectory_len}")

# Access sliced data
print(f"Partial X range: [{partial_trajectory.x.min():.2f}, {partial_trajectory.x.max():.2f}]")

## 5. Iterating Through Trajectory Points

Iterate through trajectory data points.


In [ ]:
# Iterate through first 5 trajectory points
print("First 5 trajectory points:")
for i, point in enumerate(trajectory):
    if i >= 5:
        break
    print(f"Point {i}: x={point.x:.3f}, y={point.y:.3f}")


## 6. Trajectory Batching

In [ ]:
# Create batch trajectories (3 trajectories, 20 timesteps each)
batch_size, time_steps = 3, 20
batch_x = np.random.randn(batch_size, time_steps)
batch_y = np.random.randn(batch_size, time_steps)
batch_frame = np.random.randn(batch_size, time_steps, 10)
batch_timestamps = np.tile(np.arange(time_steps) * 0.1, (batch_size, 1))

@dataclass()
class Simple2DCoordinateTrajectory(tct.BaseTrajectoryFeature):
    x: np.ndarray
    y: np.ndarray
    frame: np.ndarray
    timestamps: np.ndarray


batch_trajectory = Simple2DCoordinateTrajectory(
    feature_name="batch 2d coordinate",
    x=batch_x,
    y=batch_y,
    frame=batch_frame,
    timestamps=batch_timestamps,
    batch=True
)

print(f"Batch trajectory shape: {batch_trajectory.x.shape}")
print(f"Number of trajectories: {batch_trajectory.x.shape[0]}")
print(f"Timesteps per trajectory: {batch_trajectory.x.shape[1]}")
print(f"Trajectory length: {len(batch_trajectory)}")

print(batch_trajectory)

## 6.5. Post-Processing Callbacks

TCT provides three callback methods for custom data post-processing during instantiation:
- `on_begin_post_init_callback()` - Executed at the start
- `post_init_feature_callback(feature_name)` - Executed once per feature
- `on_exit_post_init_callback()` - Executed at the end

These callbacks enable automatic feature engineering, validation, and transformations.


In [ ]:
# Example: Trajectory with automatic feature computation
@dataclass()
class TrajectoryWithCallbacks(tct.BaseTrajectoryFeature):
    x: np.ndarray
    y: np.ndarray
    velocity: np.ndarray
    
    def on_begin_post_init_callback(self):
        """Called at the beginning - setup or initial transformations."""
        print("→ on_begin_post_init_callback executed")
        # Store trajectory metadata
        self.set_dynamic_attribute('total_points', len(self.x))
    
    def post_init_feature_callback(self, feature_name: str):
        """Called once per feature - create derived features."""
        print(f"→ post_init_feature_callback executed for: {feature_name}")
        feature = self.get_dynamic_attribute(feature_name)
        
        if isinstance(feature, np.ndarray) and feature_name in ['x', 'y']:
            # Create cumulative sum for position features
            cumsum = np.cumsum(feature)
            self.set_dynamic_attribute(f"{feature_name}_cumsum", cumsum)
    
    def on_exit_post_init_callback(self):
        """Called at the end - final computations and validation."""
        print("→ on_exit_post_init_callback executed")
        # Compute total distance traveled
        dx = np.diff(self.x, prepend=self.x[0])
        dy = np.diff(self.y, prepend=self.y[0])
        distances = np.sqrt(dx**2 + dy**2)
        self.set_dynamic_attribute('distance_per_step', distances)
        self.set_dynamic_attribute('total_distance', np.sum(distances))
        
        # Validate data
        assert len(self.x) > 0, "Trajectory must have at least one point"

# Create trajectory - callbacks execute automatically
print("\n=== Creating trajectory with callbacks ===")
callback_trajectory = TrajectoryWithCallbacks(
    feature_name="trajectory_with_callbacks",
    x=np.array([0., 1., 2., 3., 4.]),
    y=np.array([0., 1., 0., -1., 0.]),
    velocity=np.array([1., 1., 1., 1., 1.])
)

print("\n=== Accessing computed fields ===")
print(f"Total points (from on_begin): {callback_trajectory.total_points}")
print(f"X cumsum (from post_init_feature): {callback_trajectory.x_cumsum}")
print(f"Y cumsum (from post_init_feature): {callback_trajectory.y_cumsum}")
print(f"Total distance (from on_exit): {callback_trajectory.total_distance:.2f}")
print(f"Distance per step: {callback_trajectory.distance_per_step}")


### Advanced Callback Example: Feature Normalization

This example shows how to normalize features automatically during instantiation.


In [ ]:
@dataclass()
class NormalizedTrajectory(tct.BaseTrajectoryFeature):
    x: np.ndarray
    y: np.ndarray
    velocity: np.ndarray
    
    def post_init_feature_callback(self, feature_name: str):
        """Normalize all numeric features to [0, 1] range."""
        feature = self.get_dynamic_attribute(feature_name)
        
        if isinstance(feature, np.ndarray):
            min_val = np.min(feature)
            max_val = np.max(feature)
            
            if max_val > min_val:
                normalized = (feature - min_val) / (max_val - min_val)
                self.set_dynamic_attribute(f"{feature_name}_normalized", normalized)

# Create trajectory with auto-normalization
norm_trajectory = NormalizedTrajectory(
    feature_name="normalized_trajectory",
    x=np.array([10., 20., 30., 40., 50.]),
    y=np.array([100., 150., 200., 250., 300.]),
    velocity=np.array([5., 10., 15., 20., 25.])
)

print("Original vs Normalized values:")
print(f"X: {norm_trajectory.x}")
print(f"X normalized: {norm_trajectory.x_normalized}")
print(f"\nVelocity: {norm_trajectory.velocity}")
print(f"Velocity normalized: {norm_trajectory.velocity_normalized}")


For more details on post-processing callbacks, see the [Post-Processing Callbacks documentation](../documentation/post_processing_callbacks.md).


## 7. Data Validation

The dataclasses automatically validate data consistency.


In [ ]:
# Example of data validation - this will work
trajectory_length = 5
valid_x = np.arange(trajectory_length)
valid_y = np.arange(trajectory_length)
valid_frame = np.random.randn(trajectory_length, 10)
valid_timestamps = np.arange(trajectory_length) * 0.1


# Create trajectory container
valid_trajectory = Simple2DCoordinateTrajectory(
    feature_name="2D coordinate – valid",
    x=valid_x,
    y=valid_y,
    frame=valid_frame,
    timestamps=valid_timestamps
)

print("Valid trajectory created successfully!")
print(f"Trajectory length: {valid_trajectory.trajectory_len}\n")

# Demonstrate error handling with mismatched dimensions
try:
    # This should raise an error due to mismatched array lengths
    invalid_y = np.arange(trajectory_length - 1)  # 4 elements - mismatch!

    invalid_trajectory = Simple2DCoordinateTrajectory(
        feature_name="2D coordinate – invalid",
        x=valid_x,
        y=invalid_y,
        frame=valid_frame,
        timestamps=valid_timestamps
    )

except ValueError as e:
    print(f"Expected error caught: {type(e).__name__}")
    print(e)

## Summary

This notebook covered the essential usage patterns of TCT dataclasses:

1. **Basic trajectory creation**
2. **Factory-based creation**
3. **Data access** and manipulation methods
4. **Trajectory slicing** for extracting portions of data
5. **Iteration** through trajectory points
6. **Trajectory batching**
6.5. **Post-processing callbacks** for automatic feature engineering
7. **Data validation** and error handling

For more examples, refer to the documentation at `documentation/direct_instantiation.md` and `documentation/post_processing_callbacks.md`.
